In [1]:
from dem_to_hypso import dem_download
from reanalysis import reanalysis
import pandas as pd
import numpy as np

list_reanalysis      = ['era5', 'jra3qg']
formatted_reanalysis = {'era5': 'ERA5', 'jra3qg': 'JRA3QG'}
csv_user_input       = './user_input/config_globsim_pre_hypso.csv'

list_stations        = list(pd.read_csv(csv_user_input)['station_name'])

In [2]:
from netCDF4 import Dataset

In [3]:
T_pl_in = {ra: Dataset(f'/fs/yedoma/home/vpo001/globsim/DReaMIT_demo/reanalysis/{ra}/interpolated/{ra}_pl_config_globsim_with_hypso.nc')['t' if ra=='era5' else 'Temperature'] for ra in list_reanalysis}
h_pl_in = {ra: Dataset(f'/fs/yedoma/home/vpo001/globsim/DReaMIT_demo/reanalysis/{ra}/interpolated/{ra}_pl_config_globsim_with_hypso.nc')['z' if ra=='era5' else 'Geopotential height'] for ra in list_reanalysis}
T_pl_surface_in = {ra: Dataset(f'/fs/yedoma/home/vpo001/globsim/DReaMIT_demo/reanalysis/{ra}/interpolated/{ra}_pl_config_globsim_with_hypso_surface.nc')['t' if ra=='era5' else 'Temperature'] for ra in list_reanalysis}
h_pl_surface_in = {ra: Dataset(f'/fs/yedoma/home/vpo001/globsim/DReaMIT_demo/reanalysis/{ra}/interpolated/{ra}_pl_config_globsim_with_hypso_surface.nc')['height' if ra=='era5' else 'height'] for ra in list_reanalysis}
grid_elev_in = {ra: Dataset(f'/fs/yedoma/home/vpo001/globsim/DReaMIT_demo/reanalysis/{ra}/interpolated/{ra}_to_config_globsim_with_hypso.nc')['z' if ra=='era5' else 'Geopotential'] for ra in list_reanalysis}

In [4]:
print(T_pl_in['era5'].units, ',', T_pl_in['jra3qg'].units)
print(h_pl_in['era5'].units, ',', h_pl_in['jra3qg'].units)

print(T_pl_surface_in['era5'].units, ',', T_pl_surface_in['jra3qg'].units)
print(h_pl_surface_in['era5'].units, ',', h_pl_surface_in['jra3qg'].units)

print(grid_elev_in['era5'].units, ',', grid_elev_in['jra3qg'].units)

K , K
m**2 s**-2 , gpm
K , K
m , m
m**2 s**-2 , m2 s-2


In [5]:
def format_temp_elev_inversions(reanalysis: str, T_pl_in: np.ndarray, h_pl_in: np.ndarray, T_pl_surface_in: np.ndarray, h_pl_surface_in: np.ndarray, grid_elev_in: np.ndarray):
    """
    Formatting of the interpolated GlobSim values to only account for elevations between 
    maximal elevation (5000m) and minimal elevation (lowest of grid or station)
    
    Parameters
    ----------
    reanalysis : str
        Name of the reanalysis, 'era5' or 'jra3qg
    T_pl_in : np.ndarray
        Pressure-level air temperature [K] with dimensions (time, level, station)
    h_pl_in : np.ndarray
        Pressure-level elevation [m] with dimensions (time, level, station)
    T_pl_surface_in : np.ndarray
        Pressure-level air temperature at surface (station) [K] with dimensions (time, station)
    h_pl_surface_in : np.ndarray
        Pressure-level elevation at surface (station) [m] with dimensions (station,)
    grid_elev_in : np.ndarray
        Time-invariant grid elevation [m] with dimensions (time, station)

    Returns
    ------- 
    T_pl : np.ndarray
        Pressure-level air temperature [K] with dimensions (time, level, station)
        all values for elevations outside the prescribed range are nans
    h_pl : np.ndarray
        Pressure-level elevation [m] with dimensions (time, level, station)
        all values for elevations outside the prescribed range are nans
        converted to meters for ERA5
    T_pl_surface : np.ndarray
        Pressure-level air temperature at surface (station) [K] with dimensions (time, station)
        all values for elevations outside the prescribed range are nans
    h_pl_surface : np.ndarray
        Pressure-level elevation at surface (station) [m] with dimensions (station,)
        all values for elevations outside the prescribed range are nans
    grid_elev : np.ndarray
        Time-invariant grid elevation [m] with dimensions (time, station)
        all values for elevations outside the prescribed range are nans
    """
    num_station = grid_elev_in.shape[-1]
    grid_elev = float(grid_elev_in[0,0])/9.80665
    zmax = 5000 ############# !!!!!!!!!!!!!!!!!!! #############
    h_pl_in = h_pl_in[:]/(9.80665 if reanalysis=='era5' else 1)
    zmin = {station: np.min([h_pl_surface_in[station],grid_elev]) for station in range(num_station)}

    T_pl = {station: [] for station in range(num_station)}
    h_pl = {station: [] for station in range(num_station)}
    T_pl_surface = {station: [] for station in range(num_station)}
    h_pl_surface = {station: [] for station in range(num_station)}

    for station in range(num_station):
        mask_z = (h_pl_in[:,:,station] < zmin[station]) | (h_pl_in[:,:,station] > zmax)
        T_pl[station] = np.ma.array(T_pl_in[:,:,station], mask=mask_z).filled(np.nan)
        h_pl[station] = np.ma.array(h_pl_in[:,:,station], mask=mask_z).filled(np.nan)

        T_pl_surface[station] = T_pl_surface_in[:,station]
        h_pl_surface[station] = h_pl_surface_in[station]

    T_pl = np.stack(list(T_pl.values()), axis=-1) # dimensions (time, level, station)
    h_pl = np.stack(list(h_pl.values()), axis=-1) # dimensions (time, level, station)
    T_pl_surface = np.stack(list(T_pl_surface.values()), axis=-1) # dimensions (time, station)
    h_pl_surface = np.stack(list(h_pl_surface.values()), axis=-1) # dimensions (station)

    return T_pl, h_pl, T_pl_surface, h_pl_surface, grid_elev

In [80]:
grid_elev_in['era5'].shape[-1]

In [ ]:
T_pl, h_pl, T_pl_surface, h_pl_surface, grid_elev = format_temp_elev_inversions('era5', T_pl_in['era5'], h_pl_in['era5'], T_pl_surface_in['era5'], h_pl_surface_in['era5'], grid_elev_in['era5'])